## 1. Imports and keys
Loads the `typesense` library and reads your keys from the `.env` file (copy `.env.example` to `.env` and fill it in). Run this first.

In [ ]:
import os
import typesense
from dotenv import load_dotenv

load_dotenv()  # reads TYPESENSE_* and GROQ_API_KEY from .env

## 2. Connect to Typesense and create the `books_latest` collection
- `client` is the connection to your Typesense Cloud cluster.
- `book_schema` lists the fields of a book (title, authors, price, rating and so on) and their types.
- `client.collections.create(...)` creates the collection. Run it **only once**: running it again gives an "already exists" error.

In [ ]:
client = typesense.Client({
    'nodes': [{
        'host': os.environ['TYPESENSE_HOST'],
        'port': '443',
        'protocol': 'https'
    }],
    'api_key': os.environ['TYPESENSE_API_KEY'],
    'connection_timeout_seconds': 5
})

book_schema ={
    'name': 'books_latest',
    'fields': [
         {"name": "book_id", "type": "string"},
        {"name": "title", "type": "string"},
        {"name": "authors", "type": "string[]"},
        {"name": "publisher", "type": "string"},
        {"name": "publication_year", "type": "int32"},
        {"name": "edition", "type": "string"},
        {"name": "isbn_13", "type": "string", "optional": True},
        {"name": "genre", "type": "string", "facet": True},
        {"name": "subjects", "type": "string[]", "facet": True},
        {"name": "language", "type": "string", "facet": True},
        {"name": "format", "type": "string", "facet": True},
        {"name": "page_count", "type": "int32", "optional": True},
        {"name": "description", "type": "string"},
        {"name": "price", "type": "float"},
        {"name": "currency", "type": "string"},
        {"name": "average_rating", "type": "float"},
        {"name": "rating_count", "type": "int32"},
        {"name": "availability", "type": "string", "facet": True},
        {"name": "stock_count", "type": "int32"},
    ],
}

# Create the collection
client.collections.create(book_schema)

print("Books collection created successfully!")


## 3. Upload the 500 books
Reads `books_catalog_500.json`, takes the list under the `"books"` key, and imports it into `books_latest`. It prints how many books were uploaded.
Run it **only once**: running it again adds the same books a second time.

In [ ]:
import json
with open("books_catalog_500.json", "r", encoding="utf-8") as file:
    data = json.load(file)["books"]
    result = client.collections['books_latest'].documents.import_(data)
print(len(data), "books;", sum(r["success"] for r in result), "imported")


## 4. Search for a book by title or author
Searches `title` and `authors` for the text in `q` and sorts the matches by `rating_count`, highest first. Change `q` to search for something else.

In [ ]:
search_parameters = {
    'q': 'An Atlas of Paper Kingdom',
    'query_by': 'title,authors',
    'sort_by': 'rating_count:desc'
}

result = client.collections['books_latest'].documents.search(search_parameters)
print(result['found'], "results")
for hit in result['hits']:
    doc = hit['document']
    print(doc['title'], '-', doc['authors'], '-', doc['rating_count'])



## 5. List all books by rating count
`q: "*"` matches every book. The result is sorted by `rating_count` with the highest first, and the books are printed with their authors and rating counts.

In [ ]:
search_parameters={
    'q': "*",
    'query_by':"authors",
    'sort_by':"rating_count:desc"
    }
result = client.collections['books_latest'].documents.search(search_parameters)
print(result['found'], "results")
for hit in result['hits']:
    doc = hit['document']
    print(doc['title'], '-', doc['authors'], '-', doc['rating_count'])

## 6. Imports for the RAG part
Loads the LangChain tools used below: `TextLoader` (reads a text file), `Typesense` (vector store), `ChatGroq` (the LLM), `CharacterTextSplitter` (cuts text into chunks) and `HuggingFaceEmbeddings` (turns text into numbers).

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_groq import ChatGroq
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings


## 7. Check the Groq API key
`ChatGroq` reads `GROQ_API_KEY` from the environment. It was loaded from `.env` in cell 1, so never paste the key into the notebook.

In [ ]:
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in your .env file"

## 8. Load `test.txt`, split it into chunks, create embeddings
- `TextLoader` reads `test.txt`.
- `CharacterTextSplitter` cuts it into pieces of about 1000 characters (`docs`), with 100 characters of overlap so no sentence is lost at the edges.
- `HuggingFaceEmbeddings` is the model that will turn each chunk into numbers for searching. The first run downloads it, which takes a while.

In [ ]:
loader=TextLoader("test.txt")
documents= loader.load()
text_splitter=CharacterTextSplitter(chunk_size = 1000, chunk_overlap=100 )
docs=text_splitter.split_documents(documents)

embeddings= HuggingFaceEmbeddings()

## 9. Push the `test.txt` chunks to Typesense
Turns each chunk in `docs` into an embedding and stores it in a new Typesense collection called `rag_docs`. It reuses the `client` from cell 2 for the connection and key.
Run it **only once**: running it again uploads every chunk a second time. It creates `docsearch`, which the next cell uses.

In [ ]:
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client=client,
    typesense_collection_name="langchain",
)

## 10. Ask a question and find the matching text
Sets the question in `query` and searches `docsearch` (created in cell 9) for the chunks closest to it.

In [ ]:
query="what is artificial intelligence?"
found_docs=docsearch.similarity_search(query)
print(found_docs[0].page_content)
